# GPR92 / LPAR5 mapping in pancreas — cleaned analysis notebook

This notebook is part of an integrated workflow (neonatal → chronic pancreatitis → spatial → integrated spatial).

**Local data (not committed):**
- `../data/raw/`
- `../data/processed/`

Outputs are written to `../outputs/`.

In [ ]:
# --- Environment & paths (portable) ---
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# Local project modules
import sys
sys.path.append(str(Path("..").resolve()))
from src import utils, preprocessing, scoring, plotting, spatial

In [ ]:
import scanpy as sc
import pandas as pd

# Paths to your files
control_path = r"../data/raw/GSM6198712_Control.counts.tsv.gz"
case_path = r"../data/raw/GSM6198711_Case.counts.tsv.gz"

# Load expression matrices
control_df = pd.read_csv(control_path, sep="\t", index_col=0)
case_df = pd.read_csv(case_path, sep="\t", index_col=0)

# Transpose so genes are columns (Scanpy expects cells x genes)
control_df = control_df.T
case_df = case_df.T

# Create AnnData objects
adata_control = sc.AnnData(control_df)
adata_control.obs["condition"] = "Control"

adata_case = sc.AnnData(case_df)
adata_case.obs["condition"] = "Case"

# Concatenate into one AnnData
adata = adata_control.concatenate(adata_case, batch_key="sample", batch_categories=["Control", "Case"])

adata


In [ ]:
# Identify mitochondrial genes (mouse mt genes start with 'mt-')
adata.var["mt"] = adata.var_names.str.startswith("mt-")

# Compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

# Visualize distributions before filtering
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, groupby='sample')

# Scatter plots to help visualize cutoffs
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

# Apply filters based on paper
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.n_genes_by_counts < 5000, :]
adata = adata[adata.obs.pct_counts_mt < 20, :]

adata


In [ ]:
# Normalize each cell to have total counts = 1e4 (library size normalization)
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform the data
sc.pp.log1p(adata)

# Store raw counts before further processing
adata.raw = adata


In [ ]:
# Identify highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Plot HVGs
sc.pl.highly_variable_genes(adata)

# Keep only HVGs
adata = adata[:, adata.var.highly_variable]


In [ ]:
# Scale each gene to unit variance, clip extreme values to avoid outliers dominating
sc.pp.scale(adata, max_value=10)

# Run PCA
sc.tl.pca(adata, svd_solver='arpack')

# Plot variance ratio to see how many PCs to use
sc.pl.pca_variance_ratio(adata, log=True)


In [ ]:
# Build the nearest-neighbors graph using top 10 PCs (as in the paper)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=10)

# UMAP embedding
sc.tl.umap(adata)
sc.pl.umap(adata, color=['sample', 'condition'])


In [ ]:
# Cluster using Leiden (graph-based, like Seurat's Louvain method)
sc.tl.leiden(adata, resolution=0.5)  # Try resolution=0.3–1.0 if you want more/fewer clusters

# Visualize clusters on UMAP
sc.pl.umap(adata, color=['leiden'], legend_loc='on data')


In [ ]:
# Broad panel of markers across known pancreas cell types
marker_genes = [
    "Amy2a5", "Cpa1",        # Acinar
    "Krt19", "Sox9",         # Ductal
    "Ins1", "Ins2", "Pdx1",  # Beta
    "Gcg",                   # Alpha
    "Sst",                   # Delta
    "Ppy",                   # PP
    "Cd68", "Adgre1",        # Macrophage
    "Cd3d", "Cd3e",          # T cells
    "Col1a1", "Col3a1", "Dcn", # Fibroblasts
    "Pecam1", "Cdh5"         # Endothelial
]

sc.pl.umap(adata, color=marker_genes, cmap="magma", ncols=4)


In [ ]:
print(adata.var_names[:20])  # see the first 20 genes


In [ ]:
# Check if the gene exists at all (case-insensitive match)
any(adata.var_names.str.upper() == "AMY2A5")


In [ ]:
# Search for similar gene names
[gene for gene in adata.var_names if "amy" in gene.lower()]


In [ ]:
import mygene

# Initialize the query object
mg = mygene.MyGeneInfo()

# Query Ensembl IDs to get gene symbols (batch mode)
ensembl_ids = adata.var_names.tolist()
query_results = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='mouse')

# Convert to DataFrame
gene_map = pd.DataFrame(query_results).set_index('query')

# Add gene symbols to adata.var (from previous step)
adata.var['gene_symbol'] = gene_map['symbol']

# (NEW) Save current Ensembl IDs for safety
adata.var['ensembl_id'] = adata.var_names

# Ensure no duplicated symbols cause problems downstream
adata.var_names_make_unique()


In [ ]:
marker_genes = ["Amy2a5", "Sox9", "Ins1", "Gcg", "Sst", "Pecam1", "Col1a1", "Cd68", "Cd3d"]
sc.pl.umap(adata, color=marker_genes, cmap="magma", ncols=4)


In [ ]:
# Define your intended marker genes
marker_genes = ["Amy2a5", "Sox9", "Ins1", "Gcg", "Sst", "Pecam1", "Col1a1", "Cd68", "Cd3d", "Lpar5", "Timp1", "Mmp2"]

# Check which ones are found in the dataset by matching gene_symbol
valid_markers = [gene for gene in marker_genes if gene in adata.var['gene_symbol'].values]

print("Valid markers found in data:", valid_markers)


In [ ]:
sc.pl.umap(adata, color="leiden", legend_loc="on data")


In [ ]:
# Perform differential expression: one-vs-all using Wilcoxon test
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', n_genes=50)

# Plot top markers per cluster
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)


In [ ]:
"LPAR5" in adata.var['gene_symbol'].values


In [ ]:
[gene for gene in adata.var['gene_symbol'].dropna().values if "lpar5" in gene.lower()]


In [ ]:
[gene for gene in adata.var['gene_symbol'].dropna().values if "gpr" in gene.lower()]


In [ ]:
"ENSMUSG00000030503" in adata.var_names  # Lpar5's Ensembl ID in mouse


In [ ]:
control_df = pd.read_csv(control_path, sep="\t", index_col=0).T
case_df = pd.read_csv(case_path, sep="\t", index_col=0).T


In [ ]:
# Reload raw data again
import pandas as pd

control_path = r"../data/raw/GSM6198712_Control.counts.tsv.gz"
case_path = r"../data/raw/GSM6198711_Case.counts.tsv.gz"

control_raw = pd.read_csv(control_path, sep="\t", index_col=0).T
case_raw = pd.read_csv(case_path, sep="\t", index_col=0).T


In [ ]:
# Check for gene names in raw (rows are genes after transpose)
[gene for gene in control_raw.columns if "Lpar5" in gene.lower() or "Gpr92" in gene.lower()]


In [ ]:
# Search in raw data for LPAR5's Ensembl ID (mouse)
"ENSMUSG00000030503" in control_raw.columns, "ENSMUSG00000030503" in case_raw.columns


In [ ]:
# If found, print some stats
if "ENSMUSG00000030503" in control_raw.columns:
    print("Control LPAR5 expression summary:")
    print(control_raw["ENSMUSG00000030503"].describe())

if "ENSMUSG00000030503" in case_raw.columns:
    print("Case LPAR5 expression summary:")
    print(case_raw["ENSMUSG00000030503"].describe())
